# 네이버 LPOD vs 방송사 직접 트랙 비교 (Colab/로컬용)

동일 언론사(SBS/KBS/MBC) × 동일 기간에 대해 **네이버 LPOD 트랙** (`본문_bs4_{press}_*.csv`) 과 **방송사 직접 트랙** (`본문_bs4_{press}_direct_*.csv`) 의 수집 결과를 비교한다. 같은 기간 같은 base_press(SBS/KBS/MBC)의 두 트랙 결과를 자동으로 페어링.

- 입력: `data/본문_bs4_{press}_{YYMMDD}_{YYMMDD}.csv` 와 `data/본문_bs4_{press}_direct_{YYMMDD}_{YYMMDD}.csv`
- 출력: 화면 출력 + `data/비교요약_{press}_{period}.csv` (페어별 요약 표)
- 비교 항목:
  - 트랙별 건수
  - 카테고리 분포 상위 N (각자 카테고리 체계가 다름 — 네이버는 한국식 표준 분류, 직접은 방송사 자체 분류)
  - 본문 길이 분포 (평균/중앙/min/max)
  - title 정규화 후 완전 일치 매칭 — 공통/네이버 only/직접 only
  - SBS는 link 안 news_id 추출 매칭도 추가 (네이버 LPOD `055/0001353588` vs SBS_direct `news_id=N0001353588`)
- 매칭 한계: 네이버에서 제목을 줄이거나 KBS/MBC가 송고 시 다른 ID 부여하는 경우 정확 일치 매칭률 낮음. 본 노트북은 대략적인 분포 비교용

In [ ]:
# Colab 환경 세팅 — pandas만 사용 (시각화 없이 표 출력만)
!pip install -q pandas

In [ ]:
# # Colab에서 실행할 때만 아래 3줄 주석 해제 — 로컬/WSL에서는 그대로 두기
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
import os
import re
import unicodedata
from pathlib import Path

import pandas as pd

# Colab 기본 경로 — 마운트 안 돼 있으면 except로 폴백
# news/ 하위 (방송사 직접 트랙 결과 폴더)
try:
    PROJECT_DIR = Path('/content/drive/MyDrive/Text-data-Analysis_26-Spring')
    if not PROJECT_DIR.exists():
        raise FileNotFoundError
except Exception:
    PROJECT_DIR = Path('/home/carol/Text-data-Analysis_26-Spring')

os.chdir(PROJECT_DIR)
print(f'현재 작업 폴더: {Path.cwd()}')

# 본문 CSV가 모인 폴더 — 네이버 LPOD/SBS_direct/KBS_direct/MBC_direct 모두 여기에 들어감
DATA_DIR = PROJECT_DIR / 'data' / 'news'
print(f'DATA_DIR: {DATA_DIR}')


# Drive 동기화 환경에서는 한글 파일명이 NFD로 풀려 들어오는 경우가 있어 매칭 전에 NFC로 통일
def normalize_text(text):
    return unicodedata.normalize('NFC', str(text))


def normalize_name(path):
    return normalize_text(path.name)


# 본문 CSV 파일명에서 press(base + direct 여부)와 기간 분리
# 예: '본문_bs4_SBS_260501_260507.csv' -> base='SBS', is_direct=False, period='260501_260507'
# 예: '본문_bs4_KBS_direct_260505_260511.csv' -> base='KBS', is_direct=True, period='260505_260511'
FILENAME_PATTERN = re.compile(r'^본문_bs4_(.+?)(_direct)?_(\d{6})_(\d{6})\.csv$')


def parse_body_filename(path):
    name = normalize_name(path)
    m = FILENAME_PATTERN.match(name)
    if not m:
        return None
    base, direct_suffix, start, end = m.groups()
    return {
        'base_press': base,
        'is_direct': bool(direct_suffix),
        'period': f'{start}_{end}',
        'path': path,
    }


# data 폴더 스캔 → (base_press, period)별로 네이버/직접 매칭
# 비교 대상은 둘 다 존재하는 페어만
pairs = {}  # (base_press, period) -> {'naver': path, 'direct': path}
for path in DATA_DIR.iterdir():
    if not path.is_file():
        continue
    info = parse_body_filename(path)
    if not info:
        continue
    key = (info['base_press'], info['period'])
    track = 'direct' if info['is_direct'] else 'naver'
    pairs.setdefault(key, {})[track] = info['path']

matched_pairs = {k: v for k, v in pairs.items() if 'naver' in v and 'direct' in v}
only_naver = {k: v for k, v in pairs.items() if 'naver' in v and 'direct' not in v}
only_direct = {k: v for k, v in pairs.items() if 'naver' not in v and 'direct' in v}

print(f'\n매칭 페어: {len(matched_pairs)}개')
for (press, period) in sorted(matched_pairs):
    print(f'  - {press} / {period}')
if only_naver:
    print(f'\n네이버만 있는 (직접 없음): {len(only_naver)}개')
    for k in sorted(only_naver):
        print(f'  - {k[0]} / {k[1]}')
if only_direct:
    print(f'\n직접만 있는 (네이버 없음): {len(only_direct)}개')
    for k in sorted(only_direct):
        print(f'  - {k[0]} / {k[1]}')

In [ ]:
# title 매칭 정규화 — 공백/문장부호 차이로 인한 미매칭 줄임
# 네이버에서 제목 미세 수정하는 경우 있어 완전일치는 어렵지만 공백/특수문자 정리만으로도 매칭률 ↑
def normalize_title(text):
    if not isinstance(text, str):
        return ''
    # 모든 공백을 하나로, 양끝 strip
    t = re.sub(r'\s+', '', text)
    # 따옴표/괄호 정리 — 같은 의미 다른 문자(“” ' ") 통일
    t = t.replace('\u201c', '"').replace('\u201d', '"').replace('\u2018', "'").replace('\u2019', "'")
    return t


# 네이버 LPOD link에서 article ID 추출 — 'mnews/article/{oid}/{aid}' 패턴
# 예: 'https://n.news.naver.com/mnews/article/055/0001353588' -> ('055', '0001353588')
def parse_naver_link(link):
    m = re.match(r'https://n\.news\.naver\.com/mnews/article/(\d+)/(\d+)', str(link))
    return (m.group(1), m.group(2)) if m else None


# SBS 직접 link에서 news_id 추출
# 예: 'https://news.sbs.co.kr/news/endPage.do?news_id=N0001353588' -> '0001353588'
def parse_sbs_direct_link(link):
    m = re.search(r'news_id=N(\d+)', str(link))
    return m.group(1) if m else None


# 한 페어 비교 — 건수/카테고리/본문 길이/매칭
def compare_pair(base_press, period, naver_path, direct_path):
    df_naver = pd.read_csv(naver_path, encoding='utf-8-sig')
    df_direct = pd.read_csv(direct_path, encoding='utf-8-sig')

    print()
    print(f'{"="*70}')
    print(f'  {base_press} / {period}')
    print(f'{"="*70}')

    # 1) 건수
    print(f'\n[1] 건수')
    print(f'  네이버 LPOD : {len(df_naver):>6,} 건  ({naver_path.name})')
    print(f'  {base_press}_direct : {len(df_direct):>6,} 건  ({direct_path.name})')
    diff = len(df_direct) - len(df_naver)
    print(f'  차이 (직접-네이버): {diff:+,} 건')

    # 2) 카테고리 분포 (각자 다른 분류 체계라 직접 비교 불가, 분포만 봄)
    print(f'\n[2] 카테고리 분포 — 상위 10개씩')
    print(f'  네이버 LPOD:')
    for cat, n in df_naver.get('category', pd.Series(dtype=object)).value_counts().head(10).items():
        print(f'    {cat or "(빈칸)":<20s} {n:>4d}')
    print(f'  {base_press}_direct:')
    for cat, n in df_direct.get('category', pd.Series(dtype=object)).value_counts().head(10).items():
        print(f'    {cat or "(빈칸)":<20s} {n:>4d}')

    # 3) 본문 길이 분포 — 본문 정규화 차이로 정확 비교는 어려우나 대략적 양 비교
    body_naver = df_naver.get('body', pd.Series(dtype=str)).fillna('').str.len()
    body_direct = df_direct.get('body', pd.Series(dtype=str)).fillna('').str.len()
    print(f'\n[3] 본문 길이 (글자 수)')
    print(f'  {"항목":<12s}{"네이버":>12s}{"직접":>12s}')
    for label, fn in [('평균', 'mean'), ('중앙', 'median'), ('min', 'min'), ('max', 'max')]:
        v_naver = getattr(body_naver, fn)()
        v_direct = getattr(body_direct, fn)()
        print(f'  {label:<12s}{v_naver:>12.0f}{v_direct:>12.0f}')

    # 4) title 정규화 매칭 — 공통/유일
    titles_naver = df_naver.get('title', pd.Series(dtype=str)).fillna('').map(normalize_title)
    titles_direct = df_direct.get('title', pd.Series(dtype=str)).fillna('').map(normalize_title)
    set_naver = set(titles_naver)
    set_direct = set(titles_direct)
    set_naver.discard('')  # 빈 제목 제외
    set_direct.discard('')
    common = set_naver & set_direct
    only_n = set_naver - set_direct
    only_d = set_direct - set_naver
    print(f'\n[4] title 정규화(공백/따옴표 통일) 후 매칭')
    print(f'  공통       : {len(common):>5d} 건')
    print(f'  네이버 only : {len(only_n):>5d} 건')
    print(f'  직접 only   : {len(only_d):>5d} 건')
    if set_naver:
        print(f'  네이버 매칭률: {len(common)/len(set_naver)*100:.1f}%')
    if set_direct:
        print(f'  직접 매칭률 : {len(common)/len(set_direct)*100:.1f}%')

    # 5) SBS만 — link에서 추출한 news_id 기반 매칭 (네이버 oid=055와 SBS_direct news_id가 동일)
    id_overlap = None
    if base_press == 'SBS':
        ids_naver = {parse_naver_link(l)[1] for l in df_naver.get('link', []) if parse_naver_link(l) and parse_naver_link(l)[0] == '055'}
        ids_direct = {parse_sbs_direct_link(l) for l in df_direct.get('link', []) if parse_sbs_direct_link(l)}
        id_overlap = ids_naver & ids_direct
        print(f'\n[5] SBS 한정 — link news_id 기반 매칭 (가장 정확)')
        print(f'  네이버 SBS news_id : {len(ids_naver):>5d}')
        print(f'  직접 SBS news_id   : {len(ids_direct):>5d}')
        print(f'  공통(id) : {len(id_overlap):>5d}')
        if ids_naver:
            print(f'  네이버 ID 매칭률: {len(id_overlap)/len(ids_naver)*100:.1f}%')

    return {
        'base_press': base_press,
        'period': period,
        'naver_count': len(df_naver),
        'direct_count': len(df_direct),
        'title_common': len(common),
        'title_only_naver': len(only_n),
        'title_only_direct': len(only_d),
        'naver_match_rate_pct': round(len(common)/len(set_naver)*100, 1) if set_naver else 0,
        'direct_match_rate_pct': round(len(common)/len(set_direct)*100, 1) if set_direct else 0,
        'id_overlap_sbs': len(id_overlap) if id_overlap is not None else None,
        'avg_body_naver': round(body_naver.mean(), 0) if len(body_naver) else 0,
        'avg_body_direct': round(body_direct.mean(), 0) if len(body_direct) else 0,
    }

In [ ]:
# 모든 매칭 페어 순회 + 결과 누적
summary_rows = []
for (press, period) in sorted(matched_pairs):
    paths = matched_pairs[(press, period)]
    row = compare_pair(press, period, paths['naver'], paths['direct'])
    summary_rows.append(row)

if not summary_rows:
    print('비교할 매칭 페어가 없습니다. 네이버 LPOD와 방송사 직접 본문 CSV가 같은 기간으로 모두 있어야 합니다.')
else:
    print()
    print('=' * 70)
    print('  종합 요약')
    print('=' * 70)
    summary_df = pd.DataFrame(summary_rows)
    print(summary_df.to_string(index=False))

In [ ]:
# 샘플 출력 — 매칭 페어 1개 골라 공통/네이버only/직접only 기사 몇 건씩 표시
# 정성적 확인용 (어떤 기사가 매칭됐고 어떤 게 한쪽에만 있는지)
SAMPLE_PRESS = None  # 예: 'SBS' 또는 None이면 첫 페어 사용
SAMPLE_N = 5  # 각 분류별 표시 건수

if matched_pairs:
    if SAMPLE_PRESS:
        key = next((k for k in matched_pairs if k[0] == SAMPLE_PRESS), None)
    else:
        key = sorted(matched_pairs)[0]

    if key:
        press, period = key
        paths = matched_pairs[key]
        df_naver = pd.read_csv(paths['naver'], encoding='utf-8-sig')
        df_direct = pd.read_csv(paths['direct'], encoding='utf-8-sig')

        # title 정규화 후 매칭 분류
        df_naver['title_norm'] = df_naver.get('title', pd.Series(dtype=str)).fillna('').map(normalize_title)
        df_direct['title_norm'] = df_direct.get('title', pd.Series(dtype=str)).fillna('').map(normalize_title)
        set_naver = set(df_naver['title_norm']) - {''}
        set_direct = set(df_direct['title_norm']) - {''}
        common = set_naver & set_direct

        print(f'\n=== 샘플: {press} / {period} ===')
        print(f'\n[공통 기사 {SAMPLE_N}건]')
        df_common = df_naver[df_naver['title_norm'].isin(common)].head(SAMPLE_N)
        for _, r in df_common.iterrows():
            print(f'  {r.get("title", "")[:80]}')

        print(f'\n[네이버 only {SAMPLE_N}건 — 직접에 없는 제목]')
        df_only_n = df_naver[~df_naver['title_norm'].isin(common) & (df_naver['title_norm'] != '')].head(SAMPLE_N)
        for _, r in df_only_n.iterrows():
            print(f'  {r.get("title", "")[:80]}')

        print(f'\n[직접 only {SAMPLE_N}건 — 네이버에 없는 제목]')
        df_only_d = df_direct[~df_direct['title_norm'].isin(common) & (df_direct['title_norm'] != '')].head(SAMPLE_N)
        for _, r in df_only_d.iterrows():
            print(f'  {r.get("title", "")[:80]}')
else:
    print('샘플 출력할 매칭 페어가 없습니다.')

In [ ]:
# 종합 요약을 CSV로 저장 — 사후 분석/리포트용
if summary_rows:
    summary_df = pd.DataFrame(summary_rows)
    # 페어가 여러 개면 모두 한 파일에, 단일 페어면 파일명에 press_period 박음
    if len(summary_rows) == 1:
        r = summary_rows[0]
        save_path = DATA_DIR / f'비교요약_{r["base_press"]}_{r["period"]}.csv'
    else:
        save_path = DATA_DIR / '비교요약_전체.csv'
    summary_df.to_csv(save_path, index=False, encoding='utf-8-sig')
    print(f'요약 저장: {save_path}')
    print(f'행 수: {len(summary_df)}')
else:
    print('저장할 요약이 없습니다.')